# Docling PDF Parser for IR Documents

This notebook processes PDF documents from the company reports folder using Docling for intelligent document parsing and content extraction.

## Features:
- Process all PDF documents in company folders
- Extract structured content using Docling
- Parse financial tables, text, and metadata
- Generate summaries and insights
- Export results to organized JSON files


In [8]:
# Install required packages
%pip install docling docling-core transformers torch
%pip install pandas numpy matplotlib seaborn plotly
%pip install tqdm pathlib


/Users/RiyanshiKedia/Documents/GitHub/investment-report-extractor/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/Users/RiyanshiKedia/Documents/GitHub/investment-report-extractor/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/Users/RiyanshiKedia/Documents/GitHub/investment-report-extractor/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Any, Optional
from tqdm import tqdm
import logging
from datetime import datetime
import re

# Docling imports
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ All imports successful")


✅ All imports successful


## Configuration and Setup


In [15]:
# Configuration
REPORTS_DIR = Path("../data/reports")
OUTPUT_DIR = Path("../data/docling_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Docling configuration - simplified approach
try:
    # Try the new API first
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = True
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.do_cell_matching = True
    
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: pipeline_options
        }
    )
    print("✅ Using advanced pipeline options")
except Exception as e:
    print(f"⚠️ Advanced options failed: {e}")
    print("🔄 Falling back to basic configuration...")
    
    # Fallback to basic configuration
    converter = DocumentConverter()
    print("✅ Using basic configuration")

print(f"📁 Reports directory: {REPORTS_DIR.absolute()}")
print(f"📁 Output directory: {OUTPUT_DIR.absolute()}")
print(f"🔧 Docling converter initialized")


✅ Using advanced pipeline options
📁 Reports directory: /Users/RiyanshiKedia/Documents/GitHub/investment-report-extractor/notebooks/../data/reports
📁 Output directory: /Users/RiyanshiKedia/Documents/GitHub/investment-report-extractor/notebooks/../data/docling_output
🔧 Docling converter initialized


## Document Discovery and Analysis


In [16]:
def discover_documents(reports_dir: Path) -> Dict[str, List[Dict[str, Any]]]:
    """Discover all PDF documents in company folders"""
    documents = {}
    
    for company_dir in reports_dir.iterdir():
        if not company_dir.is_dir():
            continue
            
        company_name = company_dir.name
        ticker = company_name.split(' - ')[0]
        
        company_docs = []
        
        # Load metadata if available
        metadata_file = company_dir / "_metadata.json"
        metadata = {}
        if metadata_file.exists():
            with open(metadata_file, 'r') as f:
                metadata = json.load(f)
        
        # Find all PDF files
        for file_path in company_dir.glob("*.pdf"):
            doc_info = {
                'file_path': str(file_path),
                'file_name': file_path.name,
                'file_size_mb': file_path.stat().st_size / (1024 * 1024),
                'ticker': ticker,
                'company_name': company_name,
                'metadata': metadata
            }
            
            # Extract document type from filename
            filename_lower = file_path.name.lower()
            if '10-k' in filename_lower or 'annual' in filename_lower:
                doc_info['document_type'] = '10-K Annual Report'
            elif '10-q' in filename_lower or 'quarterly' in filename_lower:
                doc_info['document_type'] = '10-Q Quarterly Report'
            elif 'proxy' in filename_lower:
                doc_info['document_type'] = 'Proxy Statement'
            elif 'press' in filename_lower or 'earnings' in filename_lower:
                doc_info['document_type'] = 'Press Release'
            elif 'presentation' in filename_lower:
                doc_info['document_type'] = 'Presentation'
            else:
                doc_info['document_type'] = 'Other PDF Document'
            
            # Extract year and quarter from filename
            year_match = re.search(r'20(\d{2})', file_path.name)
            quarter_match = re.search(r'[Qq]([1-4])', file_path.name)
            
            doc_info['extracted_year'] = int(year_match.group(1)) + 2000 if year_match else None
            doc_info['extracted_quarter'] = int(quarter_match.group(1)) if quarter_match else None
            
            company_docs.append(doc_info)
        
        if company_docs:
            documents[ticker] = company_docs
    
    return documents

# Discover all documents
all_documents = discover_documents(REPORTS_DIR)

print(f"📊 Discovered {len(all_documents)} companies with PDF documents")
total_pdfs = sum(len(docs) for docs in all_documents.values())
print(f"📄 Total PDF documents: {total_pdfs}")

# Show summary
for ticker, docs in all_documents.items():
    print(f"  {ticker}: {len(docs)} PDFs")
    for doc in docs[:2]:  # Show first 2 documents
        print(f"    - {doc['document_type']} ({doc['extracted_year']})")


📊 Discovered 24 companies with PDF documents
📄 Total PDF documents: 153
  JPM: 10 PDFs
    - 10-Q Quarterly Report (2025)
    - Other PDF Document (2024)
  CAT: 7 PDFs
    - Other PDF Document (2025)
    - 10-Q Quarterly Report (2025)
  GS: 6 PDFs
    - 10-K Annual Report (2024)
    - Presentation (None)
  AXP: 5 PDFs
    - 10-K Annual Report (2024)
    - Press Release (2025)
  V: 7 PDFs
    - Other PDF Document (2025)
    - Other PDF Document (2025)
  WMT: 2 PDFs
    - Proxy Statement (2025)
    - Other PDF Document (2025)
  CSCO: 5 PDFs
    - Proxy Statement (2024)
    - Press Release (2024)
  TRV: 10 PDFs
    - Proxy Statement (2025)
    - 10-Q Quarterly Report (2025)
  UNH: 7 PDFs
    - Press Release (2025)
    - 10-Q Quarterly Report (2025)
  MSFT: 2 PDFs
    - Presentation (2024)
    - Other PDF Document (2023)
  MCD: 1 PDFs
    - Press Release (2025)
  PG: 8 PDFs
    - Proxy Statement (2025)
    - 10-K Annual Report (2025)
  HD: 7 PDFs
    - Press Release (2025)
    - 10-K Annua

In [17]:
def extract_document_content(doc_path: str) -> Dict[str, Any]:
    """Extract content from PDF using Docling with fallback options"""
    try:
        # Try with basic converter first
        try:
            result = converter.convert(doc_path)
        except Exception as e:
            if "backend" in str(e).lower():
                # Fallback: create a new basic converter
                from docling.document_converter import DocumentConverter
                basic_converter = DocumentConverter()
                result = basic_converter.convert(doc_path)
            else:
                raise e
        
        document = result.document
        
        # Extract text content
        text_content = document.export_to_markdown()
        
        # Extract tables
        tables = []
        try:
            for element in document.iterate_items():
                if hasattr(element, 'label') and element.label == 'table':
                    table_data = {
                        'caption': getattr(element, 'caption', ''),
                        'content': str(element),
                        'bbox': getattr(element, 'bbox', None)
                    }
                    tables.append(table_data)
        except Exception as e:
            logger.warning(f"Could not extract tables: {e}")
        
        # Extract metadata
        metadata = {
            'title': getattr(document, 'title', ''),
            'page_count': len(document.pages) if hasattr(document, 'pages') else 0,
            'language': getattr(document, 'language', 'en'),
            'creation_date': getattr(document, 'creation_date', None)
        }
        
        # Extract structured elements
        elements = []
        try:
            for element in document.iterate_items():
                element_info = {
                    'type': type(element).__name__,
                    'label': getattr(element, 'label', ''),
                    'text': str(element)[:200] if hasattr(element, '__str__') else '',
                    'bbox': getattr(element, 'bbox', None)
                }
                elements.append(element_info)
        except Exception as e:
            logger.warning(f"Could not extract elements: {e}")
        
        return {
            'success': True,
            'text_content': text_content,
            'tables': tables,
            'metadata': metadata,
            'elements': elements,
            'error': None
        }
        
    except Exception as e:
        logger.error(f"Error processing {doc_path}: {str(e)}")
        return {
            'success': False,
            'text_content': '',
            'tables': [],
            'metadata': {},
            'elements': [],
            'error': str(e)
        }

print("🔧 Docling processing function ready with fallback")


🔧 Docling processing function ready with fallback


In [18]:
# Process a sample of documents (first 3 companies, max 2 docs each)
sample_documents = {}
for ticker, docs in list(all_documents.items())[:3]:
    sample_documents[ticker] = docs[:2]  # Max 2 docs per company

print(f"📄 Processing sample: {len(sample_documents)} companies")
total_sample_docs = sum(len(docs) for docs in sample_documents.values())
print(f"📄 Total sample documents: {total_sample_docs}")

# Process documents
processed_results = {}

for ticker, docs in tqdm(sample_documents.items(), desc="Processing companies"):
    company_results = []
    
    for doc in tqdm(docs, desc=f"Processing {ticker}", leave=False):
        print(f"\n🔄 Processing: {doc['file_name']}")
        
        # Extract content
        result = extract_document_content(doc['file_path'])
        
        # Combine with document info
        doc_result = {
            'document_info': doc,
            'extraction_result': result
        }
        
        company_results.append(doc_result)
        
        # Show progress
        if result['success']:
            print(f"  ✅ Success: {len(result['text_content'])} chars, {len(result['tables'])} tables")
        else:
            print(f"  ❌ Failed: {result['error']}")
    
    processed_results[ticker] = company_results

print(f"\n✅ Processing complete! Processed {len(processed_results)} companies")


📄 Processing sample: 3 companies
📄 Total sample documents: 6


Processing companies:   0%|          | 0/3 [00:00<?, ?it/s]2025-10-10 04:37:58,674 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-10 04:37:58,680 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-10 04:37:58,738 - INFO - Going to convert document batch...
2025-10-10 04:37:58,740 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-10-10 04:37:58,767 - INFO - Loading plugin 'docling_defaults'
2025-10-10 04:37:58,770 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-10-10 04:37:58,778 - INFO - Loading plugin 'docling_defaults'
2025-10-10 04:37:58,784 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']



🔄 Processing: 10-Q Quarterly Report - 2025.pdf


2025-10-10 04:37:59,263 - INFO - Accelerator device: 'mps'
2025-10-10 04:38:02,946 - INFO - Accelerator device: 'mps'
2025-10-10 04:38:04,223 - INFO - Accelerator device: 'mps'
2025-10-10 04:38:04,921 - INFO - Processing document 10-Q Quarterly Report - 2025.pdf
2025-10-10 04:51:38,939 - INFO - Finished converting document 10-Q Quarterly Report - 2025.pdf in 816.21 sec.
2025-10-10 04:51:40,084 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-10 04:51:40,089 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-10 04:51:40,097 - INFO - Going to convert document batch...
2025-10-10 04:51:40,097 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-10-10 04:51:40,098 - INFO - Accelerator device: 'mps'


  ✅ Success: 1407805 chars, 0 tables

🔄 Processing: Supplemental Data - 2024 - Supplemental Financial Information (Related to the.pdf


...


🔄 Processing: Annual Report - 2024 - Goldman Sachs 2024 Annual Report.pdf


2025-10-10 05:09:25,586 - INFO - Accelerator device: 'mps'
2025-10-10 05:09:27,168 - INFO - Accelerator device: 'mps'
2025-10-10 05:09:27,879 - INFO - Processing document Annual Report - 2024 - Goldman Sachs 2024 Annual Report.pdf
2025-10-10 05:19:36,062 - INFO - Finished converting document Annual Report - 2024 - Goldman Sachs 2024 Annual Report.pdf in 613.79 sec.
2025-10-10 05:19:36,632 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-10 05:19:36,635 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-10 05:19:36,641 - INFO - Going to convert document batch...
2025-10-10 05:19:36,642 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-10-10 05:19:36,642 - INFO - Accelerator device: 'mps'


  ✅ Success: 1457347 chars, 0 tables

🔄 Processing: Presentation - Sustainability_ Our Issuance Framework.pdf


2025-10-10 05:19:39,441 - INFO - Accelerator device: 'mps'
2025-10-10 05:19:40,511 - INFO - Accelerator device: 'mps'
2025-10-10 05:19:41,009 - INFO - Processing document Presentation - Sustainability_ Our Issuance Framework.pdf
2025-10-10 05:19:48,053 - INFO - Finished converting document Presentation - Sustainability_ Our Issuance Framework.pdf in 11.42 sec.
Processing companies: 100%|██████████| 3/3 [41:49<00:00, 836.47s/it]

  ✅ Success: 9361 chars, 0 tables

✅ Processing complete! Processed 3 companies


## Export Results and Analysis


In [19]:
# Export processed results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Export full results
results_file = OUTPUT_DIR / f"docling_results_{timestamp}.json"
with open(results_file, 'w') as f:
    json.dump(processed_results, f, indent=2, default=str)

# Create summary DataFrame
summary_data = []
for ticker, docs in processed_results.items():
    for doc_result in docs:
        doc_info = doc_result['document_info']
        extraction = doc_result['extraction_result']
        
        summary_data.append({
            'ticker': ticker,
            'document_type': doc_info['document_type'],
            'year': doc_info['extracted_year'],
            'quarter': doc_info['extracted_quarter'],
            'file_name': doc_info['file_name'],
            'file_size_mb': doc_info['file_size_mb'],
            'extraction_success': extraction['success'],
            'text_length': len(extraction['text_content']) if extraction['success'] else 0,
            'table_count': len(extraction['tables']) if extraction['success'] else 0,
            'error': extraction['error'] if not extraction['success'] else None
        })

summary_df = pd.DataFrame(summary_data)
summary_csv = OUTPUT_DIR / f"document_summary_{timestamp}.csv"
summary_df.to_csv(summary_csv, index=False)

print(f"✅ Results exported:")
print(f"  📄 Full results: {results_file}")
print(f"  📋 Summary CSV: {summary_csv}")

# Display summary
print(f"\n📋 Document Processing Summary:")
print(f"Total documents processed: {len(summary_df)}")
print(f"Successful extractions: {summary_df['extraction_success'].sum()}")
print(f"Total text extracted: {summary_df['text_length'].sum():,} characters")
print(f"Total tables extracted: {summary_df['table_count'].sum()}")

display(summary_df)


✅ Results exported:
  📄 Full results: ../data/docling_output/docling_results_20251010_052017.json
  📋 Summary CSV: ../data/docling_output/document_summary_20251010_052017.csv

📋 Document Processing Summary:
Total documents processed: 6
Successful extractions: 6
Total text extracted: 5,411,288 characters
Total tables extracted: 0


,ticker,document_type,year,quarter,file_name,file_size_mb,extraction_success,text_length,table_count,error
0,JPM,10-Q Quarterly Report,2025.0,NaN,10-Q Quarterly Report - 2025.pdf,2.076263,True,1407805,0,None
1,JPM,Other PDF Document,2024.0,NaN,Supplemental Data - 2024 - Supplemental Financ...,0.213936,True,47695,0,None
2,CAT,Other PDF Document,2025.0,NaN,8-K Current Report - 2025 - 8-K dated Septembe...,4.093246,True,1998193,0,None
3,CAT,10-Q Quarterly Report,2025.0,2.0,10-Q Quarterly Report - 2025_Q2 - Caterpillar ...,3.148190,True,490887,0,None
4,GS,10-K Annual Report,2024.0,NaN,Annual Report - 2024 - Goldman Sachs 2024 Annu...,5.916259,True,1457347,0,None
5,GS,Presentation,NaN,NaN,Presentation - Sustainability_ Our Issuance Fr...,0.597182,True,9361,0,None


## Financial Data Extraction


In [20]:
def extract_financial_metrics(text_content: str) -> Dict[str, Any]:
    """Extract financial metrics from text content"""
    metrics = {}
    
    # Revenue patterns
    revenue_patterns = [
        r'revenue[\s:]*\$?([\d,]+(?:\.[\d]+)?)\s*(?:million|billion|m|b)?',
        r'total revenue[\s:]*\$?([\d,]+(?:\.[\d]+)?)\s*(?:million|billion|m|b)?',
        r'net revenue[\s:]*\$?([\d,]+(?:\.[\d]+)?)\s*(?:million|billion|m|b)?'
    ]
    
    # Profit patterns
    profit_patterns = [
        r'net income[\s:]*\$?([\d,]+(?:\.[\d]+)?)\s*(?:million|billion|m|b)?',
        r'profit[\s:]*\$?([\d,]+(?:\.[\d]+)?)\s*(?:million|billion|m|b)?',
        r'earnings[\s:]*\$?([\d,]+(?:\.[\d]+)?)\s*(?:million|billion|m|b)?'
    ]
    
    # Extract revenue
    for pattern in revenue_patterns:
        matches = re.findall(pattern, text_content, re.IGNORECASE)
        if matches:
            metrics['revenue'] = matches[0]
            break
    
    # Extract profit
    for pattern in profit_patterns:
        matches = re.findall(pattern, text_content, re.IGNORECASE)
        if matches:
            metrics['profit'] = matches[0]
            break
    
    return metrics

# Extract financial metrics from processed documents
financial_data = []

for ticker, docs in processed_results.items():
    for doc_result in docs:
        doc_info = doc_result['document_info']
        extraction = doc_result['extraction_result']
        
        if extraction['success']:
            metrics = extract_financial_metrics(extraction['text_content'])
            
            financial_data.append({
                'ticker': ticker,
                'document_type': doc_info['document_type'],
                'year': doc_info['extracted_year'],
                'quarter': doc_info['extracted_quarter'],
                'revenue': metrics.get('revenue'),
                'profit': metrics.get('profit'),
                'text_length': len(extraction['text_content'])
            })

financial_df = pd.DataFrame(financial_data)

print("💰 Financial Metrics Extracted:")
if not financial_df.empty:
    print(financial_df.to_string(index=False))
    
    # Export financial data
    financial_csv = OUTPUT_DIR / f"financial_metrics_{timestamp}.csv"
    financial_df.to_csv(financial_csv, index=False)
    print(f"\n✅ Financial metrics exported: {financial_csv}")
else:
    print("No financial metrics found in the sample documents.")


💰 Financial Metrics Extracted:
ticker         document_type   year  quarter revenue profit  text_length
   JPM 10-Q Quarterly Report 2025.0      NaN       ,      ,      1407805
   JPM    Other PDF Document 2024.0      NaN       ,   None        47695
   CAT    Other PDF Document 2025.0      NaN    None      ,      1998193
   CAT 10-Q Quarterly Report 2025.0      2.0    None      1       490887
    GS    10-K Annual Report 2024.0      NaN       ,      ,      1457347
    GS          Presentation    NaN      NaN    None   None         9361

✅ Financial metrics exported: ../data/docling_output/financial_metrics_20251010_052017.csv


## Next Steps and Usage Instructions


In [21]:
print("🚀 Docling PDF Parser Complete!")
print("\n📋 Summary of what was accomplished:")
print(f"✅ Processed documents from {len(processed_results)} companies")
print(f"✅ Successfully extracted content from {summary_df['extraction_success'].sum()} documents")
print(f"✅ Extracted {summary_df['table_count'].sum()} tables")
print(f"✅ Generated {summary_df['text_length'].sum():,} characters of text")

print("\n📁 Output files created:")
print(f"  📄 Full extraction results: {results_file}")
print(f"  📋 Document summary: {summary_csv}")
if not financial_df.empty:
    print(f"  💰 Financial metrics: {financial_csv}")

print("\n🔮 Next steps:")
print("  1. Process remaining documents (modify sample_documents to include more companies)")
print("  2. Implement advanced financial data extraction")
print("  3. Create automated report generation")
print("  4. Build comparison tools between companies")
print("  5. Implement real-time monitoring dashboard")

print("\n💡 Usage Instructions:")
print("  • To process all documents: Change sample_documents to use all_documents")
print("  • To process specific companies: Filter all_documents by ticker")
print("  • To process specific document types: Filter by document_type")
print("  • To run in production: Convert this notebook to a Python script")

print("\n🔧 Configuration Options:")
print("  • Modify pipeline_options for different extraction settings")
print("  • Adjust financial_metrics patterns for different formats")
print("  • Change OUTPUT_DIR to save results elsewhere")
print("  • Modify sample size for testing vs full processing")


🚀 Docling PDF Parser Complete!

📋 Summary of what was accomplished:
✅ Processed documents from 3 companies
✅ Successfully extracted content from 6 documents
✅ Extracted 0 tables
✅ Generated 5,411,288 characters of text

📁 Output files created:
  📄 Full extraction results: ../data/docling_output/docling_results_20251010_052017.json
  📋 Document summary: ../data/docling_output/document_summary_20251010_052017.csv
  💰 Financial metrics: ../data/docling_output/financial_metrics_20251010_052017.csv

🔮 Next steps:
  1. Process remaining documents (modify sample_documents to include more companies)
  2. Implement advanced financial data extraction
  3. Create automated report generation
  4. Build comparison tools between companies
  5. Implement real-time monitoring dashboard

💡 Usage Instructions:
  • To process all documents: Change sample_documents to use all_documents
  • To process specific companies: Filter all_documents by ticker
  • To process specific document types: Filter by docum